# Atelier Préparation de Données Tabulaires

**Contexte** : préparation d'un jeu de données de capteurs IoT issus de bâtiments intelligents
(température, humidité, CO₂, consommation énergétique, occupation...) en vue d'un modèle de
Machine Learning (prédiction de consommation / détection d'anomalies).

Ce notebook suit la structure de l'atelier, **question par question**.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")
%matplotlib inline

---
# Partie 1 – Explorer les données

### 1) Charger les données CSV

In [3]:
df = pd.read_csv("../data/smart_building_raw.csv")
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


### 2) Afficher les premières lignes du dataset

In [4]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


### 3) Afficher les dernières lignes du dataset

In [5]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


### 4) Combien d'observations contient le dataset ?

In [6]:
n_obs = df.shape[0]
print(f"Le dataset contient {n_obs} observations (lignes).")

Le dataset contient 507 observations (lignes).


### 5) Combien de variables possède le dataset ?

In [7]:
n_var = df.shape[1]
print(f"Le dataset possède {n_var} variables (colonnes).")
df.columns.tolist()

Le dataset possède 14 variables (colonnes).


['id_mesure',
 'date',
 'batiment',
 'type_batiment',
 'zone',
 'temperature',
 'humidite',
 'co2',
 'occupation',
 'consommation_kwh',
 'mode_climatisation',
 'etat_systeme',
 'jour_semaine',
 'alerte']

### 6) Identifier les variables numériques

On utilise `select_dtypes` pour repérer les colonnes de type numérique.
`id_mesure` est numérique au sens du type mais c'est en réalité un **identifiant**
(cf. question 9), on l'exclut donc des variables numériques "utiles".

In [8]:
variables_numeriques = df.select_dtypes(include=[np.number]).columns.tolist()
print("Variables numériques (type) :", variables_numeriques)

variables_numeriques_utiles = [c for c in variables_numeriques if c != "id_mesure"]
print("Variables numériques utiles (hors identifiant) :", variables_numeriques_utiles)

Variables numériques (type) : ['id_mesure', 'temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']
Variables numériques utiles (hors identifiant) : ['temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']


### 7) Identifier les variables catégorielles

Colonnes de type `object` (texte), hors la date qui est une variable temporelle à part.

In [9]:
variables_categorielles = df.select_dtypes(include="object").columns.tolist()
variables_categorielles = [c for c in variables_categorielles if c != "date"]
print("Variables catégorielles :", variables_categorielles)

Variables catégorielles : ['batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


/var/folders/0h/_d5jczk96_v7v7jl988tw0xm0000gn/T/ipykernel_91315/4155203364.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  variables_categorielles = df.select_dtypes(include="object").columns.tolist()
